<a href="https://colab.research.google.com/github/ssk-algoverse/sae-binding/blob/main/circuit/Analyse-Circuit-Sebastian.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
from huggingface_hub import hf_hub_download
from transformer_lens import HookedTransformer, HookedTransformerConfig, utils, ActivationCache
import numpy as np
import pandas as pd
import ast
from torch.utils.data import Dataset, DataLoader
import torch
from functools import partial
import circuitsvis as cv
from IPython.display import display, Markdown
import matplotlib.pyplot as plt
import plotly.express as px
import re
import einops
import plotly.graph_objects as go
from jaxtyping import Float, Int
from plotly.subplots import make_subplots
from torch import Tensor
from rich import print as rprint
from rich.table import Column, Table
from IPython.display import HTML, display
from tqdm import tqdm

# Setup

In [2]:
from huggingface_hub import hf_hub_download

REPO_ID = "sebastianhoenig/no-ln"
FILENAME = "D256_L2_H2_attnOnly1_lr2.0e-04_wd0.01_ep1328.pt"

weights_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)

D256_L2_H2_attnOnly1_lr2.0e-04_wd0.01_ep(…):   0%|          | 0.00/2.48M [00:00<?, ?B/s]

In [3]:
REPO_ID = "sojup/entity_binding_test"
FILENAME = "id_to_entity.csv"

id_mapping_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)

In [5]:
E = 100
T = 10
D_VOCAB = E + T + 3
N_LAYERS = 2
HEADS = 2
d_model = 256
n_ctx   = 64

def build_model(n_layers: int, n_heads: int) -> HookedTransformer:
    if d_model % n_heads != 0:
        return None
    d_head = d_model // n_heads

    cfg = HookedTransformerConfig(
        n_layers=n_layers,
        n_heads=n_heads,
        d_model=d_model,
        d_head=d_head,
        n_ctx=n_ctx,
        d_vocab=D_VOCAB,
        d_vocab_out=E,
        attn_only=True,
        normalization_type=None,
        positional_embedding_type="rotary",
    )
    return HookedTransformer(cfg)

model = build_model(N_LAYERS, HEADS)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Load the model
pretrained_weights = torch.load(weights_path, map_location=device, weights_only=True)
state_dict = pretrained_weights["model"]
model.load_state_dict(state_dict)

print("Model loaded successfully.")

# need this for attention patching later
model.cfg.use_attn_result = True

Moving model to device:  cpu
Model loaded successfully.


In [6]:
id_mapping_df = pd.read_csv(id_mapping_path)
id_to_entity = dict(zip(id_mapping_df['id'], id_mapping_df['name']))

In [7]:
id_to_entity[100] = 'loves'
id_to_entity[101] = 'works with'
id_to_entity[102] = 'interacts with'
id_to_entity[103] = 'lives with'
id_to_entity[104] = 'has a grudge against'
id_to_entity[105] = 'is interested in'
id_to_entity[106] = 'plays with'
id_to_entity[107] = 'goes to school with'
id_to_entity[108] = 'is jealous of'
id_to_entity[109] = 'wants'

In [8]:
id_to_entity_rev = {v: k for k, v in id_to_entity.items()}

In [9]:
class EntityBindingDataset(Dataset):
    def __init__(self, dataframe, parse_tokens_if_str=True):
        self.df = dataframe.reset_index(drop=True)
        self.parse_tokens_if_str = parse_tokens_if_str

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        seq = row["tokens"]
        tokens = torch.tensor(seq, dtype=torch.long)
        label  = torch.tensor(int(row["label"]), dtype=torch.long)
        return tokens, label

In [10]:
from datasets import load_dataset

dataset = load_dataset("sojup/entity_binding", split="test")

In [11]:
test_df = dataset.to_pandas()
test_dataset = EntityBindingDataset(test_df)

In [12]:
example, label = test_dataset[0]
example, label, [id_to_entity[i.item()] for i in example], id_to_entity[label.item()]

(tensor([ 10, 104,   1, 110,  20, 103,  30, 110,  70, 105,  37, 110,  30, 100,
          24, 110, 105,  70, 111]),
 tensor(37),
 ['Jeffery',
  'has a grudge against',
  'Angel',
  ',',
  'Linda',
  'lives with',
  'Lindsay',
  ',',
  'Jason',
  'is interested in',
  'Christine',
  ',',
  'Lindsay',
  'loves',
  'Susan',
  ',',
  'is interested in',
  'Jason',
  '?'],
 'Christine')